In [0]:
import base64, hashlib, json, zlib

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("source_update_id", "")
dbutils.widgets.text("silver_update_id", "")
dbutils.widgets.text("scratch_prefix", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SOURCE_UPDATE_ID = dbutils.widgets.get("source_update_id").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
SCRATCH_PREFIX = dbutils.widgets.get("scratch_prefix").strip()
assert RUN.startswith("dq4_omop_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SOURCE_UPDATE_ID and SILVER_UPDATE_ID and SCRATCH_PREFIX
LANE = "themis"
PAYLOAD = """eNrVWv9v2jgU/1c8pFPhRhGw61ahMalbmcauR6eW2zRdT55JDERN7Mx22HHT/vd7dhJwSGC0TdiOX7b6Pb+vH7/3nOSvrzVnTp1b7Lm1Xk3NaeDJXkiF5Kw3o8ylAjucOTRUwIGlIswlwq01a1MSeP4S9qRk+HfKRUDgLyDfkjmLidYiZTOPUViWn334UxExoworMvH14il26aLFAx5ixw1asRFrNof7UaBl5swCnpAIEsha7+u3phaOFQ1Cnygt9npwMXg1Rg6PmKr/2kBn10hxRXws+BfZvGEISMSn0qF1GQX1V2fXA/ThzWCEcnrQ6HKMhiNUPz1pP2ui05Mn3QYaa9YOGlzAtjYajM4bTdQ2WgJKZCSoaxTdsNdXl3+gT8bJT5te3mg/ReRTLHkkHLpKBSxLuqDCU0vYOCWRr4D2rmuCqeMGWVMiot+aO/I48YSaYxfCobyAaoekJxVlyk6jxxQVDMKyIjvLdR4hmJH0Jp4PhpSWyKxdFWVxSYmoZzU10KO+Wcd8ig1J77d+l1eoHnAGW1IGNLw22R/9eXGBzkbnyJCL5Gb3NQoku2S5XS4Qi6Tae7IyS8IfxOpqgLKac8ZlYmZTq8VvRi1OoOhTG70L4kcUb6C0SuxmbLKhW/P5F6gaEwCq9qRzetoGchSG6SIOPca037Vuu/sUaIwbhBsWZFiQJxGEAMWcSETsmIeUmfg3EdMBjXHtRELAOTYZqzcatZLPTTbXGgWA3yzhOdIe5lGe5XqBtK8/da3kE0nFgiiPMwyKPO72oNsJhUERNAIbbDramAtoDh6blYOzvPY85ljk+9XUyLx27bJBFSS4gBpHRtPLrEV5RXfNdefeudbCljhGGJ4TiTnLFBhBp1QfNc90SAg5hQ5JpbRHncxqKWWmwpQnOvVYkx7uUo4nCm/YxeD1GL291KNSYuD58Ho8HMF/1lr3xUADce3HZdbmPgrXfx0OJoxjDvJ8Eh5mdiqlLnwYjt8gU6+gmUA26zqeSWJWMWzuPud6Xwyo5LerZgAvtXl9Mqvv4G+gy/eDq9is1e/d2dV4OB5C2l9+RFamM23mHLYBeaflzV2mWvIM0ENBF9is7g3PG3bvQxhry41ZEj3vp7S7nckkyRWdh4UnPYW5E08dDu3pKM5oHOuKG+Sm7kPWyli31RHjBetwPLBybjp3qPylV1uYQPG/VPBDXOu/n8hazFL5/X5TjT3k5mh91C5z1Hloxrv3zPgXT809hgvbSuGsU/6V6kee5Aqmnk130MKef/ZoIFvGm4U93pi2sGjl6s7LwfjDwPi1/5Ce+2nZhQLWTbLCcuRSRTz/8K0k1nv4NhLrzXeTZL2am5Wt4hDZnN7igmNeWGD07DwTmery4O6yLbe1TaOqay5uq0BXbtJbbOUyD9WqAQByd1aoooJm6lOxsf1iV++KsvY9UAZ50w+/BJXcX1D5UyAssSlzpisEWbG6HM7CVhHHj8ZYwhom+Mob2d/q4SGKWNJ/A6KAUcYWHObW/6M6lGtNII8yI0jZ2Pgflh6XEjXvcUa1o3pEwqunditIzASB6Tpi3uco93Tw3hXHKK4IBgnpOPlv/jHd/ik2ZlYae31NnVAII129dqlyTtwS95pZNyNala8s3dZaTwOmxLCVfaXywANphG8/iemz3Lgu2/cS176XxG8NbVMLWk9pbw07dwNL0jN8zmaYTKFW4zSjByvhd7jl1hT3qdD1AeK4lPrV4frdYEe/AEZ8iqRPnFtEJpKLiUSSBPRYU+JsCjoDH4S5w6HVWSjzBpO/gb7IZv8xGo7Gg6v3ZxeAzfOzj1U+M9HdoRi9Cbg1eN3tl+rd4K0InA6MMxj8oD2fO/Fte3p7mMl1pbqgpq2MqWxWdVqWjlyV8HPUkofTlfPI2TmZpmYgX6Mna1Y/60RVHz0IvvDg7PZkSB1AgRbCA5grDvLRVaK8ACJrcyp/MhuKVpG21fdX7Y3PajSC6k6r+FGu04rDlwy1R+8SF4/K/Y4mkQq2Z/D1BPoPv43CmDU1ETkaXRmL+9u8rghmU+Loq7Nv0CznXtiTyyCgSnjO/h3yAUjL6T/kRUekmLC+beyU9UQ45xma7iw5eX6RXHu22NlH04L1rt6kT4KIJRrW9U9vStct1gI5W+R38vK7W+R3qmqebpAI7EkYbfz4KkaYi0Gk1NGzoQtgJtDVSelFcm3G96c6+g8caQWI0ZsSK0HiSes3rTUlLqCrTCKfiKXFszhptVH32fHrwcvj7tMy5zjLFlNTL4a/D9AR2PTL0bZXBfqdWM5IU0+zZu4SsI4bZiT5vHD7edOCHiPL6NTNPvCaLe14S0dv2X8QWBlRUWFNYQc3EIYhWWZ62xuU9/+GKJVcSR0tTEM7n7m905Bae9ckPMkl4e//AFH1+AU="""
checks = json.loads(zlib.decompress(base64.b64decode(PAYLOAD)).decode())

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

for seq, check in enumerate(checks):
    sql = check["sql_template"].replace("{RUN}", RUN)
    sha = hashlib.sha256(sql.encode()).hexdigest()
    name = "themis:" + check["check_id"]
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
      ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'attempted',
       NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        row = spark.sql(sql).first().asDict()
        total = int(row["total_rows"] or 0)
        measured = int(row["measured_rows"] or 0)
        status = "fail" if measured > 0 else "pass"
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_check_result VALUES
          ({qs(RUN)},{qs(check['check_id'])},{qs(status)},{measured},{total},
           false,NULL,NULL,map('stage','dq4_omop_themis'),'DQ4',current_timestamp())""")
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
          ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'ok',
           NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        missing = any(x in msg for x in ("TABLE_OR_VIEW_NOT_FOUND","UNRESOLVED_COLUMN",
                                         "UNRESOLVED_FIELD","UNRESOLVED_ROUTINE"))
        if missing:
            spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_check_result VALUES
              ({qs(RUN)},{qs(check['check_id'])},'skip',0,0,false,NULL,NULL,
               map('stage','dq4_omop_themis','reason',{qs(msg[:1000])}),
               'DQ4',current_timestamp())""")
            spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
              ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'skipped_missing_table',
               {qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        else:
            spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
              ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'error',
               {qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
            raise
print({"checks":len(checks),"status":"ok"})